## Importing Libraries

In [1]:
import os
import shutil
import logging
from concurrent.futures import ThreadPoolExecutor, as_completed
from multiprocessing import cpu_count
import pandas as pd
from tqdm import tqdm

## Path Variables

In [2]:
# ---------------- CONFIG ----------------
excel_path = r"D:\Payment Reminder Notices - 2018\May 26\LRN\LRN Format May 26.xlsx"       # Excel file
output_folder = r"D:\Payment Reminder Notices - 2018\May 26\LRN\LRN"             # Folder to save PDFs
signature_path = r"D:\Payment Reminder Notices - 2018\May 26\LRN\Sign Of Advocate.png"      # Signature
background_path = r"D:\Payment Reminder Notices - 2018\May 26\LRN\Advocate BG.png"
temp_output_folder = r"D:\Payment Reminder Notices - 2018\May 26\LRN\TEMP"

## Change Batch Size Alone

In [3]:
# Parallel & batching
MAX_WORKERS = max(1, cpu_count() - 1)
BATCH_SIZE = 3000

## Generation Never Change

In [4]:
# Logging / failure
os.makedirs(output_folder, exist_ok=True)
os.makedirs(temp_output_folder, exist_ok=True)
logging.basicConfig(filename='pdf_generation.log', level=logging.INFO,
                    format='%(asctime)s %(levelname)s %(message)s')
failure_log_path = "failed_rows.csv"

# ---------------- TEMPLATE ----------------
LETTER_BODY = """«Sl_No»

<b>«Name_of_the_Borrower»</b>

<b>«Borrower_Mobile_Number»</b>

<b>REFERENCE: Loan Recall Notice and Demand Notice for Outstanding dues Rs. «Close_Out_Amount_»/- in respect of the Loan Account No. «LAN»</b>

Dear Sir/Madam,  

Under instructions from and on behalf of our client CapFloat Financial Services Private Limited (Operating under the brand name <b>“axio”</b>), having its Registered Office at Gokaldas Platinum, New No.3, (Old No.211), Bellary Road, Sadashivanagar, Bengaluru – 560080, Karnataka, India, we hereby serve you with the following notice as under: 

1)  Our client has advised us that you had availed of Online Checkout Loan-Retail Financial Facilities from our client under the Loan Account No. «LAN» and entered into a contract with our client. For the aforesaid facility being sanctioned and granted to you, you had executed the Loan Agreement and other relevant documents in favour of our client guaranteeing and assuring to repay the loan strictly in terms of the Loan Agreement. 

2)  Our client has advised us that in terms of the Contract it had paid and disbursed the Loan Amount of Rs. «Sanctioned_Amount»/- to you under the said Loan Account and you were under contractual and legal liability to repay the Loan Amount in Equated Monthly Installments. Our client has also advised us that the regular & timely payment of the Equated Monthly Installments was the essence of the contract. 

3)  Our client has advised us that you have defaulted in making the regular and timely payment of equated monthly installments in terms of the loan agreement and the same were not paid in time despite repeated requests, reminders and personal visits made by the officials of our client. Due to your financial indiscipline and delinquency to pay the equated monthly installments on time a considerable amount of is overdue and outstanding against you in your Loan Account. 

4)  Our client has advised us that it has also apprised you that if you fail to pay the equated monthly installments on time and regularize or close your account by paying the overdue amount then it shall be constrained to take recourse to appropriate legal remedy. However despite the requests and reminders of our client you have neglected to pay the outstanding dues and regularize or towards closure of your Loan account.

5)  It is stated that you have dishonestly induced and deceived our client into giving financial assistance in form of <b>«Product»</b> Loan facility with the illegal motive of securing unlawful pecuniary gains.

6)  It is stated that in the facts and circumstances as explained herein above you along with other accomplices have committed criminal offence of Cheating, which is punishable with imprisonment <b>up to 7 (Seven) years</b> under Section 420 of the Indian Penal code. That you have intentionally misappropriated the loan amount which is punishable offence under section 403 Indian Penal Code with imprisonment of two years or fine or with both and is also in violation of the end use contractual clause. You have also committed offence of criminal breach of trust under section 406 Indian Penal Code, 1860. Our client reserves its right to prosecute you and other accomplices for committing serious criminal offences of cheating, and the bank also reserve right to get the FIR registered against you and initiate appropriate action for your prosecution under Section 406, 405, 420 read with 120 B & 34 of Indian Penal Code. It is stated that the entire contract between you and our client is vitiated in view of fraud played by you addressees therefore you addressees are under legal obligation to repay the entire sum of Rs. «Close_Out_Amount_»/- along with Interest and other charges (as agreed) that has been misutilized by you under the above said Loan Agreement.

7)  Compelled by your indifferent attitude and financial discipline our client has been constrained to exercise its right under the Loan Agreement for recalling the amount under your Loan account and as such our client has decided to recall the Loan facility granted to you. Our client has stated that a sum of Rs. «Close_Out_Amount_»/- is outstanding against your Loan account as on <b>«Close_Out_Amount_Date»</b> and you are under the contractual and legal obligation to pay this sum of Rs. «Close_Out_Amount_»/-, along with penal interest, late payment charges and other charges in accordance with the terms and conditions of the Loan Agreement and such other amount pending as on date of realization of the amount. 

In view of the facts and circumstances, as explained herein above, we acting under the authority from and instructions of our client, we therefore by this notice call upon you, on behalf of our client, to make payment of the Total outstanding amount of Rs. <b>«Close_Out_Amount_»/</b>, within SEVEN DAYS of receipt of this notice. Please note that if you fail to pay aforesaid sum then our client shall be constrained to initiate further legal action, against you solely at your cost and peril.

Kindly ignore this notice if the amount demanded above is already paid. You are further called upon to pay Rs.2,000 /- to our client as cost towards the present Legal Notice. A copy of this notice is retained in our office for further necessary legal action and for future record and reference.
"""

# ---------------- PDF BACKGROUND ----------------
def add_background(canvas, doc):
    if background_path and os.path.exists(background_path):
        from reportlab.lib.pagesizes import A4
        page_width, page_height = A4
        canvas.saveState()
        canvas.drawImage(background_path, 0, 0, width=page_width, height=page_height, preserveAspectRatio=True, mask='auto')
        canvas.restoreState()

# ---------------- WORKER FUNCTION ----------------
def generate_pdf_from_row(row_dict, temp_output_folder, signature_path):
    try:
        from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Image
        from reportlab.lib.pagesizes import A4
        from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
        from reportlab.lib.enums import TA_JUSTIFY, TA_LEFT, TA_RIGHT
        from reportlab.lib.units import mm

        styles = getSampleStyleSheet()
        styles.add(ParagraphStyle(name='Justify', alignment=TA_JUSTIFY, leading=14))
        styles.add(ParagraphStyle(name='Left', alignment=TA_LEFT, leading=14))
        styles.add(ParagraphStyle(name='Right', alignment=TA_RIGHT, leading=14))

        # Extract fields
        sl_no = str(row_dict.get("Sl_No", "")).strip()
        name = str(row_dict.get("Name_of_the_Borrower", "")).strip()
        mobile = str(row_dict.get("Borrower_Mobile_Number", "")).strip()
        lan = str(row_dict.get("LAN", "")).strip()
        sanctioned = str(row_dict.get("Sanctioned_Amount", "")).strip()
        close_out_amount = str(row_dict.get("Close_Out_Amount", "")).strip()
        close_out_date = str(row_dict.get("Close_Out_Amount_Date", "")).strip()
        product = str(row_dict.get("Product", "")).strip()

        # Replace placeholders
        letter_text = LETTER_BODY.replace("«Sl_No»", sl_no) \
                                 .replace("«Name_of_the_Borrower»", name) \
                                 .replace("«Borrower_Mobile_Number»", mobile) \
                                 .replace("«LAN»", lan) \
                                 .replace("«Sanctioned_Amount»", sanctioned) \
                                 .replace("«Close_Out_Amount_»", close_out_amount) \
                                 .replace("«Close_Out_Amount_Date»", close_out_date) \
                                 .replace("«Product»", product)

        filename = f"{lan}.pdf" if lan else f"row_{sl_no}.pdf"
        tmp_path = os.path.join(temp_output_folder, filename)

        # Reserve 29 mm at the top for advocate name & logo
        reserved_top_space = 17 * mm  

        doc = SimpleDocTemplate(
            tmp_path,
            pagesize=A4,
            rightMargin=72,
            leftMargin=72,
            topMargin=72 + reserved_top_space,  # Push body down
            bottomMargin=72
        )

        story = []

        for para in letter_text.split("\n\n"):
            if para.strip():
                story.append(Paragraph(para.strip(), styles['Justify']))
                story.append(Spacer(1, 12))

        # Signature image on RIGHT
        if signature_path and os.path.exists(signature_path):
            img = Image(signature_path, width=120, height=50, hAlign='RIGHT')
            story.append(img)
            story.append(Spacer(1, 12))

        story.append(Paragraph("Authorised Signatory", styles['Right']))

        doc.build(story, onFirstPage=add_background, onLaterPages=add_background)

        return True, lan, tmp_path

    except Exception as e:
        return False, str(row_dict.get("LAN", row_dict.get("Sl_No", "unknown"))), str(e)

# ---------------- BATCH RUNNER ----------------
def run_in_batches(df, temp_output_folder, output_folder, signature_path,
                   max_workers=MAX_WORKERS, batch_size=BATCH_SIZE):
    rows = df.to_dict(orient="records")
    failures = []

    for batch_start in range(0, len(rows), batch_size):
        batch = rows[batch_start: batch_start + batch_size]

        with ThreadPoolExecutor(max_workers=max_workers) as executor:
            futures = {executor.submit(generate_pdf_from_row, row, temp_output_folder, signature_path): row for row in batch}

            for fut in tqdm(as_completed(futures), total=len(futures),
                            desc=f"Batch {batch_start+1}-{min(batch_start+batch_size,len(rows))}"):
                row = futures[fut]
                try:
                    success, lan_or_id, result = fut.result()
                    if success:
                        final_path = os.path.join(output_folder, os.path.basename(result))
                        shutil.move(result, final_path)
                    else:
                        logging.error(f"Failed {lan_or_id}: {result}")
                        failures.append({**row, "error": result})
                except Exception as e:
                    logging.exception("Unhandled exception in future")
                    failures.append({**row, "error": str(e)})

    if failures:
        pd.DataFrame(failures).to_csv(failure_log_path, index=False)
        print(f"Completed with {len(failures)} failures. See {failure_log_path} and pdf_generation.log")
    else:
        print("Completed successfully with zero failures.")

# ---------------- MAIN ----------------
if __name__ == "__main__":
    df = pd.read_excel(excel_path)
    df.columns = [col.strip().replace(" ", "_").replace("-", "_") for col in df.columns]

    if 'Close_Out_Amount_Date' in df.columns:
        df['Close_Out_Amount_Date'] = pd.to_datetime(df['Close_Out_Amount_Date'], errors='coerce').dt.strftime("%d-%b-%Y").fillna('')

    os.makedirs(temp_output_folder, exist_ok=True)
    os.makedirs(output_folder, exist_ok=True)

    run_in_batches(df, temp_output_folder, output_folder, signature_path,
                   max_workers=MAX_WORKERS, batch_size=BATCH_SIZE)


Batch 6001-6947: 100%|███████████████████████████████████████████████████████████████| 947/947 [02:27<00:00,  6.41it/s]

Completed successfully with zero failures.
